# GraphGenerator on ZINC

This notebook demonstrates the two-stage `GraphGenerator`: first generate a new interpretation graph, then instantiate base molecules conditionally from nearby ZINC examples.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
from nsppk import NSPPK
from sklearn.ensemble import RandomForestClassifier

from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_ml.feasibility import FeasibilityEstimator, FeasibilityEstimatorFeatureCannotExist
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator


In [ ]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 128
min_num_nodes = 10
max_num_nodes = 14

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


In [ ]:
cycle_tree = add(
    compose(name("cycle"), cycle()),
    compose(name("tree"), tree()),
)
decomposition_function = compose(intersection_edges(), cycle_tree)
nbits = 14
label_mode = "histogram_values"

interpretation_graphs = [
    graph_to_abstract_graph(
        graph,
        decomposition_function=decomposition_function,
        nbits=nbits,
        label_mode=label_mode,
    ).interpretation_graph.copy()
    for graph in graphs
]

def label_counts(graph):
    return Counter(data.get("label") for _, data in graph.nodes(data=True))

print("first interpretation label counts:", label_counts(interpretation_graphs[0]))


In [ ]:
edge_vectorizer = NSPPK(radius=1, distance=3, connector=1, nbits=12, parallel=True)
edge_graph_estimator = GraphEstimator(
    transformer=edge_vectorizer,
    estimator=RandomForestClassifier(
        n_estimators=80,
        random_state=0,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
)

feasibility_kwargs = dict(nbits=14, parallel=True, backend="loky", n_jobs=-1)
interpretation_feasibility = FeasibilityEstimator([
    FeasibilityEstimatorFeatureCannotExist(decomposition_function=edge(), **feasibility_kwargs),
])

edge_generator = EdgeGenerator(
    partial_feasibility_estimator=interpretation_feasibility,
    final_feasibility_estimator=interpretation_feasibility,
    graph_estimator=edge_graph_estimator,
    n_negative_per_positive=3,
    n_replicates=2,
    beam_size=3,
    max_restarts=2,
    fit_n_jobs=-1,
    fit_backend="loky",
    seed=0,
)

conditional_vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)
conditional_generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=conditional_vectorizer,
    n_jobs=1,
    debug=False,
)

generator = GraphGenerator(
    edge_generator=edge_generator,
    conditional_generator=conditional_generator,
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    seed=0,
)
generator.store(graphs, interpretation_graphs=interpretation_graphs)


In [ ]:
generated_graphs = generator.sample(
    n_samples=2,
    n_interpretation_neighbors=24,
    n_conditional_neighbors=24,
    n_instances_per_sample=2,
    interpretation_edge_removal_size=0.35,
    random_state=7,
    conditional_generate_kwargs=dict(
        random_state=7,
        max_backtracks=2000,
        max_attempts_per_sample=6,
        require_signature_coverage=True,
    ),
)

print(f"generated molecules: {len(generated_graphs)}")
print("seed indices:", generator.last_sampled_indices_)
display_graphs([graphs[i] for i in generator.last_sampled_indices_], mols_per_row=2)
display_graphs(generated_graphs, mols_per_row=4)


In [ ]:
def draw_interpretation_graphs(graphs_to_draw, titles):
    fig, axes = plt.subplots(1, len(graphs_to_draw), figsize=(4 * len(graphs_to_draw), 3))
    if len(graphs_to_draw) == 1:
        axes = [axes]
    for ax, graph, title in zip(axes, graphs_to_draw, titles):
        pos = nx.spring_layout(graph, seed=0)
        labels = {node: data.get("label", node) for node, data in graph.nodes(data=True)}
        nx.draw_networkx(graph, pos=pos, labels=labels, ax=ax, node_size=900, font_size=8)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()

draw_interpretation_graphs(
    generator.last_generated_interpretation_graphs_,
    [f"generated interpretation {i}" for i in range(len(generator.last_generated_interpretation_graphs_))],
)


In [ ]:
rows = []
for sample_id, target_graph in enumerate(generator.last_generated_interpretation_graphs_):
    seed_idx = generator.last_sampled_indices_[sample_id]
    seed_counts = label_counts(generator.stored_interpretation_graphs_[seed_idx])
    target_counts = label_counts(target_graph)
    for instance_id, generated_graph in enumerate(generated_graphs[sample_id * 2:(sample_id + 1) * 2]):
        final_interpretation = graph_to_abstract_graph(
            generated_graph,
            decomposition_function=decomposition_function,
            nbits=nbits,
            label_mode=label_mode,
        ).interpretation_graph
        rows.append({
            "sample": sample_id,
            "instance": instance_id,
            "seed_counts": seed_counts,
            "target_counts": target_counts,
            "final_counts": label_counts(final_interpretation),
        })

rows
